In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/shopping_trends.csv")

X = pd.read_csv("../data/clustering_features.csv")
X_scaled = np.load("../data/X_scaled.npy")

reference_labels = np.load("../data/reference_labels.npy")

In [2]:
profile = X.copy()
profile["Cluster"] = reference_labels

cluster_size = (
    profile["Cluster"]
    .value_counts()
    .sort_index()
)

cluster_percentage = (
    cluster_size / len(profile) * 100
).round(2)

cluster_overview = pd.DataFrame({
    "Customers": cluster_size,
    "Percentage": cluster_percentage
})

cluster_overview

,Customers,Percentage
Cluster,,
0,1986,50.92
1,1914,49.08


In [3]:
behavior_features = [
    "Purchase Amount (USD)",
    "Previous Purchases",
    "Purchase Frequency",
    "Is_Subscribed",
    "Log_Annual_Spend",
    "Log_Purchase_Engagement"
]

behavior_profile = (
    profile
    .groupby("Cluster")[behavior_features]
    .mean()
    .round(2)
)

behavior_profile

,Purchase Amount (USD),Previous Purchases,Purchase Frequency,Is_Subscribed,Log_Annual_Spend,Log_Purchase_Engagement
Cluster,,,,,,
0,58.44,23.89,4.37,0.26,5.12,4.04
1,61.14,26.87,31.07,0.28,7.36,6.39


In [4]:
age_profile = (
    profile
    .groupby("Cluster")["Age"]
    .agg(["mean", "median", "std"])
    .round(2)
)

age_profile

,mean,median,std
Cluster,,,
0,44.20,44.0,15.33
1,43.93,44.0,15.08


In [ ]:
gender_profile = pd.crosstab(
    profile["Cluster"],
    df["Gender"],
    normalize="index"
).round(3) * 100

gender_profile

Gender,Female,Male
Cluster,,
0,31.5,68.5
1,32.5,67.5


In [6]:
category_profile = pd.crosstab(
    profile["Cluster"],
    df["Category"],
    normalize="index"
).round(3) * 100

category_profile

Category,Accessories,Clothing,Footwear,Outerwear
Cluster,,,,
0,31.8,44.8,15.2,8.3
1,31.8,44.3,15.6,8.4


In [7]:
subscription_profile = pd.crosstab(
    profile["Cluster"],
    df["Subscription Status"],
    normalize="index"
).round(3) * 100

subscription_profile

Subscription Status,No,Yes
Cluster,,
0,73.6,26.4
1,72.4,27.6


In [8]:
promotion_profile = pd.crosstab(
    profile["Cluster"],
    df["Discount Applied"],
    normalize="index"
).round(3) * 100

promotion_profile

Discount Applied,No,Yes
Cluster,,
0,56.8,43.2
1,57.2,42.8


In [9]:
profiling_summary = pd.DataFrame({
    "Customers": cluster_overview["Customers"],
    "Share (%)": cluster_overview["Percentage"],
    "Avg Age": age_profile["mean"],
    "Avg Purchase": behavior_profile["Purchase Amount (USD)"],
    "Avg Previous Purchases": behavior_profile["Previous Purchases"],
    "Purchase Frequency": behavior_profile["Purchase Frequency"],
    "Subscription Rate (%)": subscription_profile["Yes"],
    "Discount Rate (%)": promotion_profile["Yes"]
})

profiling_summary.round(2)

,Customers,Share (%),Avg Age,Avg Purchase,Avg Previous Purchases,Purchase Frequency,Subscription Rate (%),Discount Rate (%)
Cluster,,,,,,,,
0,1986,50.92,44.20,58.44,23.89,4.37,26.4,43.2
1,1914,49.08,43.93,61.14,26.87,31.07,27.6,42.8


In [10]:
profiling_summary

,Customers,Share (%),Avg Age,Avg Purchase,Avg Previous Purchases,Purchase Frequency,Subscription Rate (%),Discount Rate (%)
Cluster,,,,,,,,
0,1986,50.92,44.20,58.44,23.89,4.37,26.4,43.2
1,1914,49.08,43.93,61.14,26.87,31.07,27.6,42.8


La segmentation révèle surtout deux niveaux d'intensité d'achat:

Cluster 0 : fréquence d'achat relativement faible.
Cluster 1 : fréquence d'achat nettement plus élevée.

En revanche, les deux segments sont très similaires concernant l'âge, le montant d'un achat, l'abonnement, les remises et la composition commerciale.

Cela est cohérent avec ce qu'on avait observé dans la validation : la structure détectée par K-Means est principalement portée par la fréquence et l'intensité d'achat, plutôt que par un profil démographique ou commercial distinct.